#### 교차검증과 최적의 파라미터 찾기
- 머신러닝을 사용할 때 모델의 정확도를 측정하기 위해 반드시 활용해야 하는 방법
- 딥러닝시에는 데이터의 크기가 크므로 이 방법은 사용할 필요가 없다.
- 머신러닝: 사람의 개입이 필요(특성을 추출할 때), 딥러닝: 사람의 개입이 필요없다.

In [65]:
import pandas as pd

In [66]:
wine = pd.read_csv('data/wine.csv')
wine.head()

,alcohol,sugar,pH,class
0,9.4,1.9,3.51,0.0
1,9.8,2.6,3.20,0.0
2,9.8,2.3,3.26,0.0
3,9.8,1.9,3.16,0.0
4,9.4,1.9,3.51,0.0


In [67]:
data = wine[['alcohol', 'sugar', 'pH']].to_numpy()
target = wine['class'].to_numpy()

#### 검증 세트 추가
- 19번에서 훈련세트와 테스트세트만 가지고 작업을 했지만, 테스트 세트 작업에 파라미를 조절하여 정확성을 높이면 실전 사용시 문제가 발생한다.
- 이런 문제를 바잊하기 위해 훈련/검증/테스트로 구분하여 작업을 진행한다.

In [68]:
# 전체세트중 훈련/테스트를 8:2 기준으로 분리
from sklearn.model_selection import train_test_split
train_input, test_input, train_target, test_target = \
    train_test_split(
        data,
        target,
        test_size = 0.2,
        random_state = 42
    )

In [69]:
# 훈련세트 중 훈련/검증을 8:2 기준으로 분리
sub_input, val_input, sub_target, val_target = \
    train_test_split(
        train_input,
        train_target,
        test_size = 0.2,
        random_state=42
    )

In [70]:
# 훈련, 검증, 테스트 크기 구하기
print('Train: ', sub_input.shape)
print('Valid: ', val_input.shape)
print('Test: ', test_input.shape)

Train:  (4157, 3)
Valid:  (1040, 3)
Test:  (1300, 3)


In [71]:
# 훈련과 검증으로 결정트리 모델 만들기
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier(
    min_impurity_decrease = 0.005
)
dt.fit(sub_input, sub_target)
print('Train: ', dt.score(sub_input, sub_target))
print('Valid: ', dt.score(val_input, val_target))
print('Test: ',  dt.score(test_input, test_target))

Train:  0.8611979793120038
Valid:  0.8576923076923076
Test:  0.8515384615384616


#### 교차검증
- 교차검증의 한 파트를 폴드라고 하며 기본 폴드는 5다
- 훈련과 검증을 교차하여 정확도를 구한다.
- 전체에 대한 정확도는 해당 값들의 평균으로 구한다.

In [72]:
from sklearn.model_selection import cross_validate

In [73]:
scores = cross_validate(dt, train_input, train_target)
scores['test_score'].mean()

np.float64(0.8531855704449545)

#### Optuma
- 최적화 알고리즘 기반 탐색
- 정밀하고 효율적인 Hyper Parameter 탐색
  

In [80]:
# !pip install optuna

In [86]:
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.datasets import load_iris

In [82]:
iris = load_iris()

train_input, test_input, train_target, test_target = \
    train_test_split(
        iris.data,
        iris.target,
        test_size = 0.2,
        random_state = 42,
        stratify=iris.target
    )

In [83]:
def find_param(trial):
    # 탐색할 파라미터 정의
    n_estimators =  trial.suggest_int('n_estimator', 50, 200)
    max_depth =  trial.suggest_int('max_depth', 2, 20)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 5)
    bootstrap = trial.suggest_categorical('bootstrap', [True, False])
    clf = RandomForestClassifier(
        n_estimators = n_estimators,
        max_depth = max_depth,
        min_samples_split = min_samples_split,
        min_samples_leaf = min_samples_leaf,
        bootstrap = bootstrap,
        random_state = 42
    )
    return cross_val_score(clf, train_input, train_target, cv=5).mean()

In [ ]:
study = optuna.create_study(direction='maximize')
study.optimize(find_param, n_trials=50)
print('최적의 파라미터: ', study.best_params)

In [89]:
rf = RandomForestClassifier(
        n_estimators = 70,
        max_depth = 15,
        min_samples_split = 2,
        min_samples_leaf = 2,
        bootstrap = False,
        random_state = 42
    )